# EPFL Hackathon 2026 - Data Preprocesing

In [85]:
import numpy as np
import pandas as pd

from qiskit.circuit.library import QAOAAnsatz
from qiskit_aer import Aer
from qiskit_aer.primitives import Estimator 
from qiskit import transpile
from scipy.optimize import minimize

In [86]:
# 1) Load dataset
df = pd.read_csv("underwriting_dataset.csv")
df.columns = df.columns.str.strip()

# Use "v_i" directly (expected value of investigating)
v = df["v_i"].astype(float).to_numpy()
n = len(df)

# Choose exactly K to investigate (adjust as you want)
K = min(8, n)  # investigate 8 out of 24
lam = 1.0      # network synergy weight
rho = 2000.0    # penalty weight for enforcing sum x_i = K

In [87]:
# 2) Build correlation/network matrix W (sparse, interpretable)
W = np.zeros((n, n), dtype=float)

def add_links(col, weight):
    vals = df[col].to_numpy()
    for i in range(n):
        for j in range(i + 1, n):
            if vals[i] == vals[j]:
                W[i, j] += weight
                W[j, i] += weight

add_links("garage_id",      1.0)
add_links("device_id",      1.0)
add_links("broker_id",      0.5)
add_links("ring_id_latent", 2.0)
add_links("region_id",      0.2)

# Optional: sparsify further by keeping only edges with weight >= threshold
threshold = 0.5
W[W < threshold] = 0.0

In [88]:
# 3) Build QUBO
qp = QuadraticProgram("fraud_alert_prioritization")

for i in range(n):
    qp.binary_var(f"x_{i}")

linear = {}
quadratic = {}

def add_q(u, v, c):
    key = tuple(sorted((u, v)))
    quadratic[key] = quadratic.get(key, 0.0) + float(c)

# (A) -sum v_i x_i
for i in range(n):
    linear[f"x_{i}"] = linear.get(f"x_{i}", 0.0) - float(v[i])

# (B) -lam * sum W_ij x_i x_j
for i in range(n):
    for j in range(i + 1, n):
        if W[i, j] != 0:
            add_q(f"x_{i}", f"x_{j}", -lam * float(W[i, j]))

# (C) penalty rho*(sum x - K)^2
# Expand: rho*(sum x)^2 - 2*rho*K*(sum x) + const; and (sum x)^2 = sum x + 2*sum_{i<j} x_i x_j
for i in range(n):
    linear[f"x_{i}"] = linear.get(f"x_{i}", 0.0) + rho * 1.0
    linear[f"x_{i}"] = linear.get(f"x_{i}", 0.0) - 2.0 * rho * K

for i in range(n):
    for j in range(i + 1, n):
        add_q(f"x_{i}", f"x_{j}", 2.0 * rho)

qp.minimize(linear=linear, quadratic=quadratic)

qubo = QuadraticProgramToQubo().convert(qp)

In [89]:
# 4) Solve with QAOA
H, offset = qubo.to_ising()

reps = 2
ansatz = QAOAAnsatz(cost_operator=H, reps=reps)

estimator = Estimator()

def energy(theta):
    job = estimator.run([ansatz], [H], [theta])
    return float(job.result().values[0])

theta0 = np.random.default_rng(0).uniform(0, 2*np.pi, size=ansatz.num_parameters)
res_opt = minimize(energy, theta0, method="COBYLA", options={"maxiter": 200})
theta_star = res_opt.x

# Build final circuit and sample bitstrings
qc = ansatz.assign_parameters(theta_star)
qc.measure_all()

backend = Aer.get_backend("aer_simulator")
tqc = transpile(qc, backend)
counts = backend.run(tqc, shots=3000).result().get_counts()

best = max(counts, key=counts.get)

# reverse bitstring to align with x_0..x_{n-1}
bits = np.array([int(b) for b in best[::-1]], dtype=int)  
x = bits[:n] 
selected_idx = np.where(x == 1)[0]

In [90]:
# 5) Report 
value = float(np.sum(v * x))
synergy = 0.0
for i in range(n):
    for j in range(i + 1, n):
        synergy += W[i, j] * x[i] * x[j]
synergy = float(synergy)

print("Most frequent bitstring:", best)
print("Selected cases:", selected_idx.tolist())
print("Selected count:", int(x.sum()), "(target K =", K, ")")
print("Sum individual value:", value)
print("Network synergy term:", lam * synergy)
print("Total (value + synergy):", value + lam * synergy)

display(df.loc[selected_idx].reset_index(drop=True))

Most frequent bitstring: 000101111101110000011110
Selected cases: [1, 2, 3, 4, 10, 11, 12, 14, 15, 16, 17, 18, 20]
Selected count: 13 (target K = 8 )
Sum individual value: 3208.054528326851
Network synergy term: 67.10000000000001
Total (value + synergy): 3275.154528326851


,case_id,claim_type,region_id,claim_amount,policy_age_days,prior_claims,broker_id,garage_id,device_id,review_minutes,p_fraud,ring_id_latent,severity_L,review_cost,v_i
0,1,auto,2,3251.947554,293,1,7,1,5,41,0.435422,3,975.584266,41.0,192.635015
1,2,auto,1,5774.281211,3029,0,3,8,5,66,0.308571,2,1732.284363,66.0,227.992626
2,3,auto,0,4160.428011,545,0,5,1,17,56,0.284220,1,1248.128403,56.0,139.108453
3,4,auto,4,1974.703798,1914,1,5,8,9,68,0.360451,1,592.411139,68.0,49.444431
4,10,health,3,3854.166571,2700,0,1,6,17,47,0.257497,2,770.833314,47.0,62.168196
5,11,auto,3,1764.713060,1097,0,7,5,26,84,0.230656,3,529.413918,84.0,-16.838100
6,12,auto,4,3759.536216,2598,0,0,3,25,52,0.290947,2,1127.860865,52.0,128.481259
7,14,auto,3,20918.088114,1168,1,4,8,4,74,0.561679,2,6275.426434,74.0,1864.625098
8,15,auto,1,2585.788851,1493,1,0,1,28,63,0.349341,3,775.736655,63.0,86.047991
9,16,health,2,2553.285587,1601,0,3,3,4,38,0.222623,2,510.657117,38.0,24.526337
